# NULL in Aggregation

## Overview
NULL values interact with aggregate functions in ways that are often surprising and can silently distort results. Understanding NULL behaviour in aggregation is essential for producing accurate financial and analytical reports.

**The core rule:** Almost every aggregate function *ignores* NULL values. The exceptions are `COUNT(*)` (counts all rows regardless) and set operations.

| Function | NULL behaviour |
|---|---|
| `COUNT(*)` | Counts all rows — NULLs included |
| `COUNT(col)` | Counts only non-NULL values |
| `SUM(col)` | Skips NULLs; returns NULL if *all* values are NULL |
| `AVG(col)` | Skips NULLs in both numerator AND denominator |
| `MIN(col)` / `MAX(col)` | Skips NULLs |
| `GROUP BY col` | Groups all NULLs together as one group |

**Three questions to ask whenever NULLs appear in data you're aggregating:**
1. Do NULLs represent *missing data* (should be 0 or imputed) or *not applicable* (correctly excluded)?
2. Will skipping NULLs in AVG inflate or deflate the result?
3. Will `SUM` returning NULL (instead of 0) break a downstream calculation or report?

---

In [ ]:
import sqlite3
import pandas as pd

conn = sqlite3.connect(':memory:')
conn.executescript("""
CREATE TABLE customers (customer_id INTEGER PRIMARY KEY, first_name TEXT, last_name TEXT, segment TEXT, province TEXT);
CREATE TABLE accounts (account_id INTEGER PRIMARY KEY, customer_id INTEGER, account_type TEXT, province TEXT, opened_date TEXT, status TEXT);
CREATE TABLE transactions (txn_id INTEGER PRIMARY KEY, account_id INTEGER, txn_date TEXT, txn_type TEXT, amount REAL, category TEXT, flagged INTEGER);
CREATE TABLE loans (loan_id INTEGER PRIMARY KEY, customer_id INTEGER, loan_type TEXT, principal REAL, rate_pct REAL, term_months INTEGER, issued_date TEXT, status TEXT);
INSERT INTO customers VALUES
  (1,'Aroha','Ngata','Retail','NB'),(2,'Liam','Chen','SME','BC'),
  (3,'Fatima','Al-Rashid','Wealth','ON'),(4,'James','MacLeod','Retail','NB'),
  (5,'Sofia','Petrov','SME','BC'),(6,'Noah','Williams','Retail','AB'),
  (7,'Mei','Zhang','Wealth','ON'),(8,'Ethan','Tremblay','Retail','QC'),
  (9,'Priya','Nair','SME','BC'),(10,'Marcus','Okafor','Retail','ON');
INSERT INTO accounts VALUES
  (101,1,'Chequing','NB','2019-03-15','Active'),(102,1,'Savings','NB','2019-03-15','Active'),
  (103,2,'Chequing','BC','2020-07-01','Active'),(104,2,'Investment','BC','2021-01-10','Active'),
  (105,3,'Chequing','ON','2018-05-20','Active'),(106,3,'Investment','ON','2018-05-20','Active'),
  (107,3,'Savings','ON','2022-11-01','Active'),(108,4,'Chequing','NB','2015-09-30','Active'),
  (109,4,'Loan','NB','2020-04-01','Closed'),(110,5,'Chequing','BC','2021-06-15','Active'),
  (111,5,'Savings','BC','2021-06-15','Suspended'),(112,6,'Chequing','AB','2017-11-22','Active'),
  (113,7,'Investment','ON','2016-03-08','Active'),(114,7,'Chequing','ON','2016-03-08','Active'),
  (115,8,'Chequing','QC','2023-01-05','Active'),(116,9,'Chequing','BC','2022-08-19','Active'),
  (117,9,'Investment','BC','2022-08-19','Active'),(118,10,'Chequing','ON','2020-12-01','Active'),
  (119,10,'Savings','ON','2020-12-01','Active');
INSERT INTO transactions VALUES
  (1001,101,'2023-01-05','Deposit',4200.00,'Payroll',0),(1002,101,'2023-01-12','Withdrawal',-850.00,'Rent',0),
  (1003,101,'2023-01-20','Withdrawal',-120.00,'Groceries',0),(1004,101,'2023-02-05','Deposit',4200.00,'Payroll',0),
  (1005,101,'2023-02-18','Withdrawal',-980.00,'Rent',0),(1006,102,'2023-01-15','Deposit',500.00,'Transfer',0),
  (1007,102,'2023-03-10','Interest',12.50,'Interest',0),(1008,103,'2023-01-08','Deposit',15000.00,'Payroll',0),
  (1009,103,'2023-01-25','Withdrawal',-3200.00,'Rent',0),(1010,103,'2023-02-08','Deposit',15000.00,'Payroll',0),
  (1011,104,'2023-01-31','Deposit',5000.00,'Transfer',0),(1012,105,'2023-01-10','Deposit',22000.00,'Payroll',0),
  (1013,105,'2023-01-28','Withdrawal',-8500.00,'Rent',0),(1014,105,'2023-02-10','Deposit',22000.00,'Payroll',0),
  (1015,106,'2023-02-01','Deposit',50000.00,'Investment',0),(1016,108,'2023-01-06','Deposit',3100.00,'Payroll',0),
  (1017,108,'2023-01-19','Withdrawal',-700.00,'Rent',0),(1018,108,'2023-02-06','Deposit',3100.00,'Payroll',0),
  (1019,110,'2023-01-15','Deposit',8500.00,'Payroll',0),(1020,110,'2023-02-01','Withdrawal',-2100.00,'Rent',0),
  (1021,112,'2023-01-20','Deposit',3800.00,'Payroll',0),(1022,112,'2023-02-10','Fee',-25.00,'Fee',1),
  (1023,113,'2023-01-05','Deposit',75000.00,'Investment',0),(1024,114,'2023-01-12','Deposit',12000.00,'Payroll',0),
  (1025,114,'2023-02-12','Deposit',12000.00,'Payroll',0),(1026,115,'2023-01-10','Deposit',2800.00,'Payroll',0),
  (1027,115,'2023-01-28','Withdrawal',-650.00,'Groceries',0),(1028,116,'2023-01-18','Deposit',9200.00,'Payroll',0),
  (1029,117,'2023-02-05','Deposit',10000.00,'Investment',0),(1030,118,'2023-01-22','Deposit',4500.00,'Payroll',0),
  (1031,118,'2023-02-15','Withdrawal',-1100.00,'Utilities',0),(1032,119,'2023-02-20','Interest',18.75,'Interest',0),
  (1033,101,'2023-03-05','Deposit',4200.00,'Payroll',NULL),(1034,103,'2023-03-08','Deposit',15000.00,'Payroll',0),
  (1035,112,'2023-03-15','Withdrawal',-450.00,'Groceries',1);
INSERT INTO loans VALUES
  (201,1,'Personal',15000.00,7.5,36,'2021-06-01','Current'),(202,2,'Mortgage',480000.00,4.2,300,'2020-07-15','Current'),
  (203,3,'HELOC',100000.00,6.1,60,'2019-02-28','Current'),(204,4,'Auto',28000.00,5.9,60,'2020-04-01','Paid Off'),
  (205,5,'Mortgage',390000.00,4.8,300,'2021-06-20','Current'),(206,6,'Personal',8000.00,9.2,24,'2022-03-10','Delinquent'),
  (207,7,'Mortgage',750000.00,3.9,300,'2018-09-01','Current'),(208,8,'Auto',22000.00,6.5,48,'2023-01-15','Current'),
  (209,9,'Personal',12000.00,8.1,36,'2022-08-25','Current'),(210,10,'Auto',35000.00,5.4,60,'2021-03-01','Current'),
  (211,3,'Personal',25000.00,6.8,48,'2020-11-15','Paid Off'),(212,7,'HELOC',200000.00,5.5,60,'2022-06-01','Current');
""")
conn.commit()

# Show which transactions have NULL flagged
print('Transactions with NULL in flagged column:')
q = 'SELECT txn_id, account_id, txn_type, amount, flagged FROM transactions WHERE flagged IS NULL'
print(pd.read_sql(q, conn).to_string(index=False))
print()
print('Total transactions:', conn.execute('SELECT COUNT(*) FROM transactions').fetchone()[0])
print('flagged IS NULL:  ', conn.execute('SELECT COUNT(*) FROM transactions WHERE flagged IS NULL').fetchone()[0])
print('flagged = 0:      ', conn.execute('SELECT COUNT(*) FROM transactions WHERE flagged = 0').fetchone()[0])
print('flagged = 1:      ', conn.execute('SELECT COUNT(*) FROM transactions WHERE flagged = 1').fetchone()[0])


---
## COUNT(*) vs COUNT(col) — where they diverge

In [ ]:
# Demonstrate the COUNT gap clearly
print('=== COUNT(*) vs COUNT(flagged) — and what the gap means ===')
q = """
SELECT
    COUNT(*)                AS count_all_rows,
    COUNT(flagged)          AS count_flagged_not_null,
    COUNT(*) - COUNT(flagged) AS null_flagged_rows,
    COUNT(CASE WHEN flagged = 1 THEN 1 END) AS explicitly_flagged,
    COUNT(CASE WHEN flagged = 0 THEN 1 END) AS explicitly_clear
FROM transactions
"""
print(pd.read_sql(q, conn).to_string(index=False))
print()
print('The 1 NULL flagged row is:')
print('  - Included in COUNT(*)')
print('  - Excluded from COUNT(flagged)')
print('  - Excluded from both flagged=1 and flagged=0 counts')
print('  - A silent gap in your audit trail if undetected')

# Per account impact
print()
print('=== Per account: where does COUNT(*) vs COUNT(flagged) diverge? ===')
q2 = """
SELECT account_id,
       COUNT(*)       AS total_txns,
       COUNT(flagged) AS flagged_col_count,
       COUNT(*) - COUNT(flagged) AS null_gap
FROM   transactions
GROUP BY account_id
HAVING COUNT(*) - COUNT(flagged) > 0
"""
print(pd.read_sql(q2, conn).to_string(index=False))


---
## AVG and NULL — the silent denominator shift

In [ ]:
# Construct a scenario where NULLs in a value column affect AVG
conn.executescript("""
CREATE TABLE monthly_balances (
    account_id  INTEGER,
    month       TEXT,
    balance     REAL      -- NULL = account not yet opened or data missing
);
INSERT INTO monthly_balances VALUES
  (101,'2023-01',4500.00),(101,'2023-02',3800.00),(101,'2023-03',5200.00),
  (102,'2023-01',  500.00),(102,'2023-02', NULL),(102,'2023-03',  650.00),
  (103,'2023-01',NULL),(103,'2023-02',12000.00),(103,'2023-03',14500.00),
  (108,'2023-01',2400.00),(108,'2023-02',2800.00),(108,'2023-03', NULL);
""")
conn.commit()

print('=== AVG with NULLs vs treating NULL as 0 ===')
q = """
SELECT
    account_id,
    COUNT(*)                                AS months,
    COUNT(balance)                          AS months_with_data,
    ROUND(AVG(balance), 2)                  AS avg_excl_nulls,
    ROUND(SUM(COALESCE(balance,0)) * 1.0
          / COUNT(*), 2)                    AS avg_treat_null_as_zero,
    ROUND(SUM(balance), 2)                  AS total_balance
FROM monthly_balances
GROUP BY account_id
"""
print(pd.read_sql(q, conn).to_string(index=False))
print()
print('account 102: AVG excludes the NULL month — avg is over 2 months, not 3.')
print('If NULL means the account had zero balance, avg_treat_null_as_zero is correct.')
print('If NULL means data was not collected, avg_excl_nulls is correct.')
print('The right answer depends on what NULL means in your domain.')


---
## SUM returning NULL — the all-NULL group

In [ ]:
# When every value in a group is NULL, SUM returns NULL (not 0)
conn.executescript("""
CREATE TABLE fee_waivers (
    account_id INTEGER,
    month      TEXT,
    waived_amt REAL   -- NULL = no waiver applied
);
INSERT INTO fee_waivers VALUES
  (101,'2023-01',15.00),(101,'2023-02',NULL),(101,'2023-03',NULL),
  (102,'2023-01',NULL),(102,'2023-02',NULL),(102,'2023-03',NULL),
  (103,'2023-01',25.00),(103,'2023-02',30.00),(103,'2023-03',NULL);
""")
conn.commit()

print('=== SUM returns NULL when all group values are NULL ===')
q = """
SELECT
    account_id,
    SUM(waived_amt)                    AS sum_raw,
    COALESCE(SUM(waived_amt), 0)       AS sum_safe,
    COUNT(waived_amt)                  AS months_with_waivers
FROM fee_waivers
GROUP BY account_id
"""
print(pd.read_sql(q, conn).to_string(index=False))
print()
print('account 102: all waived_amt are NULL → SUM returns NULL, not 0.')
print('COALESCE(SUM(...), 0) converts NULL to 0 for reporting.')


---
## NULL in GROUP BY — NULLs form their own group

In [ ]:
# GROUP BY treats all NULLs as equal — they form a single group
print('=== NULL category forms its own group in GROUP BY ===')
q = """
SELECT
    category,
    COUNT(*)               AS txn_count,
    ROUND(SUM(amount), 2)  AS total_amount
FROM transactions
GROUP BY category
ORDER BY total_amount DESC
"""
result = pd.read_sql(q, conn)
print(result.to_string(index=False))
print()
print('The NULL category row groups all transactions where category IS NULL.')

# Use COALESCE to give the NULL group a meaningful label
print()
print('=== COALESCE to label the NULL group ===')
q2 = """
SELECT
    COALESCE(category, '(uncategorised)') AS category,
    COUNT(*)               AS txn_count,
    ROUND(SUM(amount), 2)  AS total_amount
FROM transactions
GROUP BY COALESCE(category, '(uncategorised)')
ORDER BY total_amount DESC
"""
print(pd.read_sql(q2, conn).to_string(index=False))

conn.close()


---
## Common Pitfalls

**1. Assuming COUNT(*) and COUNT(col) are the same**
They differ whenever the column has NULL values — which is common in real data. Use `COUNT(col)` when you want the count of *populated* values, `COUNT(*)` when you want the total row count. The gap between them tells you how many NULLs are in the column.

**2. AVG silently shrinks the denominator**
`AVG(balance)` over 12 months where 3 are NULL computes the average over 9 months. If those NULLs represent zero-balance months, the average is overstated. Decide explicitly whether NULLs mean "zero" (use `COALESCE`) or "not observed" (exclude — the default).

**3. SUM of all-NULL group returns NULL, breaking downstream arithmetic**
`SUM(waived_amt)` for a group where all values are NULL returns NULL. If you then do `total_fees - SUM(waived_amt)`, the result is NULL even though total_fees is known. Always wrap aggregates that may return NULL with `COALESCE(SUM(col), 0)` in financial reports.

**4. NULL in GROUP BY creates a "mystery group"**
A GROUP BY result with a NULL label is often not what you expect — it silently lumps all records with missing category/type/status into one group. Use `COALESCE(col, 'Unknown')` in the GROUP BY expression to make the null group visible and explicitly named.

**5. Flagged-rate calculations are wrong when flagged has NULLs**
`SUM(flagged) * 1.0 / COUNT(*)` gives a wrong rate because `SUM(flagged)` skips NULLs (treating them as zero) while `COUNT(*)` includes them. Use `SUM(COALESCE(flagged, 0)) * 1.0 / COUNT(*)` to be explicit, or filter out NULLs first and document what they mean.

**6. HAVING on an aggregate silently excludes NULL-producing groups**
`HAVING AVG(balance) > 1000` excludes any group where `AVG(balance)` is NULL (all NULLs in the group). These groups vanish from the result with no error. Add `COUNT(balance) > 0` as a companion condition if you need to audit which groups had no data at all.


---
*sql_methods_library - Samantha McGarrigle*